In [7]:
# """Walk the rich package and print an AST dump for every module-level function."""

# from __future__ import annotations

# import ast
# import importlib
# import inspect
# import pkgutil
# from collections.abc import Iterator

# import rich


# def _iter_rich_module_names() -> Iterator[str]:
#     """Yields fully qualified submodule names under the rich package."""
#     yield rich.__name__
#     for _finder, name, _ispkg in pkgutil.walk_packages(
#         path=rich.__path__,
#         prefix=f"{rich.__name__}.",
#     ):
#         yield name


# def _functions_defined_in_module(module_name: str) -> Iterator[tuple[str, object]]:
#     """Yields (name, function) for functions whose __module__ matches module_name."""
#     module = importlib.import_module(module_name)
#     for attr_name, obj in inspect.getmembers(module, inspect.isfunction):
#         if getattr(obj, "__module__", None) == module_name:
#             yield attr_name, obj


# def _ast_dump_for_function(func: object) -> str | None:
#     """Returns indented ast.dump for the function source, or None if source is unavailable."""
#     try:
#         source = inspect.getsource(func)
#     except (OSError, TypeError):
#         return None
#     tree = ast.parse(source)
#     return ast.dump(tree, indent=2)


# console = rich.get_console()

# seen_ids: set[int] = set()
# total_functions = 0
# skipped_no_source = 0
# failed_imports: list[str] = []

# for module_name in _iter_rich_module_names():
#     try:
#         for func_name, func_obj in _functions_defined_in_module(module_name):
#             func_id = id(func_obj)
#             if func_id in seen_ids:
#                 continue
#             seen_ids.add(func_id)
#             total_functions += 1
#             qualname = f"{module_name}.{func_name}"
#             dump = _ast_dump_for_function(func_obj)
#             console.rule(f"[bold cyan]{qualname}[/bold cyan]")
#             if dump is None:
#                 skipped_no_source += 1
#                 console.print(
#                     f"[yellow]No Python source (builtin, dynamic, or C extension): "
#                     f"{qualname}[/yellow]"
#                 )
#             else:
#                 console.print(dump)
#     except Exception as exc:  # noqa: BLE001 — best-effort over whole package
#         failed_imports.append(module_name)
#         console.print(f"[red]Skip import {module_name!r}: {exc!r}[/red]")

# console.rule("[bold green]summary[/bold green]")
# console.print(f"{total_functions=}")
# console.print(f"{skipped_no_source=}")
# if failed_imports:
#     console.print(f"{failed_imports=}")

In [9]:
# """Walk the rich package and print AST dumps for Rich-defined functions, classes, and methods."""

# from __future__ import annotations

# import ast
# import importlib
# import inspect
# import pkgutil
# import textwrap
# from collections.abc import Iterator

# import rich


# def _iter_rich_module_names() -> Iterator[str]:
#     """Yield fully qualified submodule names under the rich package."""
#     yield rich.__name__
#     for _finder, name, _ispkg in pkgutil.walk_packages(
#         path=rich.__path__,
#         prefix=f"{rich.__name__}.",
#     ):
#         yield name


# def _module_source(module_name: str) -> str | None:
#     """Return source code for a module, or None if unavailable."""
#     module = importlib.import_module(module_name)
#     try:
#         source = inspect.getsource(module)
#     except (OSError, TypeError):
#         return None
#     return source


# def _iter_module_defs(
#     tree: ast.Module,
# ) -> Iterator[tuple[str, ast.AST]]:
#     """
#     Yield top-level Rich-defined items from a parsed module:
#       - module-level functions
#       - classes
#       - methods inside those classes
#     """
#     for node in tree.body:
#         if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
#             yield node.name, node

#         elif isinstance(node, ast.ClassDef):
#             yield node.name, node

#             for child in node.body:
#                 if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
#                     yield f"{node.name}.{child.name}", child


# def _ast_dump(node: ast.AST) -> str:
#     """Return an indented AST dump for a node."""
#     return ast.dump(node, indent=2)


# console = rich.get_console()

# total_modules = 0
# total_items = 0
# skipped_no_source = 0
# failed_imports: list[str] = []

# for module_name in _iter_rich_module_names():
#     try:
#         total_modules += 1
#         source = _module_source(module_name)
#         if source is None:
#             skipped_no_source += 1
#             console.rule(f"[bold yellow]{module_name}[/bold yellow]")
#             console.print(
#                 f"[yellow]No Python source for module: {module_name}[/yellow]"
#             )
#             continue

#         tree = ast.parse(source, filename=module_name)

#         for qualname, node in _iter_module_defs(tree):
#             total_items += 1
#             full_name = f"{module_name}.{qualname}"
#             console.rule(f"[bold cyan]{full_name}[/bold cyan]")
#             console.print(_ast_dump(node))

#     except Exception as exc:  # noqa: BLE001
#         failed_imports.append(module_name)
#         console.print(f"[red]Skip import {module_name!r}: {exc!r}[/red]")

# console.rule("[bold green]summary[/bold green]")
# console.print(f"{total_modules=}")
# console.print(f"{total_items=}")
# console.print(f"{skipped_no_source=}")
# if failed_imports:
#     console.print(f"{failed_imports=}")

In [48]:
from __future__ import annotations

import ast
import importlib
import inspect
import pkgutil
from collections.abc import Iterator
from dataclasses import dataclass, field
from pathlib import Path

from tqdm import tqdm
import rich


@dataclass
class FunctionCallInfo:
    module_name: str
    qualname: str                   # e.g. "get_console", "Console.print"
    full_name: str                  # e.g. "rich.get_console", "rich.console.Console.print"
    kind: str                       # function | method | async_function | async_method
    lineno: int | None
    end_lineno: int | None
    calls: list[str] = field(default_factory=list)  # resolved rich-only callees


@dataclass
class ModuleIndex:
    functions: dict[str, str] = field(default_factory=dict)   # local name -> full rich name
    classes: dict[str, str] = field(default_factory=dict)     # local class -> full rich name
    methods: dict[str, dict[str, str]] = field(default_factory=dict)  # class -> method -> full rich name


def _iter_rich_module_names() -> Iterator[str]:
    yield rich.__name__
    for _finder, name, _ispkg in pkgutil.walk_packages(
        path=rich.__path__,
        prefix=f"{rich.__name__}.",
    ):
        yield name


def _module_source(module_name: str) -> str | None:
    module = importlib.import_module(module_name)
    try:
        return inspect.getsource(module)
    except (OSError, TypeError):
        pass

    try:
        filename = inspect.getsourcefile(module)
        if filename is None:
            return None
        return Path(filename).read_text(encoding="utf-8")
    except OSError:
        return None


def _iter_defined_functions(
    tree: ast.Module,
) -> Iterator[tuple[str, str, str | None, ast.FunctionDef | ast.AsyncFunctionDef]]:
    """
    Yield:
      (qualname, kind, class_name, node)

    Example:
      ("get_console", "function", None, node)
      ("Console.print", "method", "Console", node)
    """
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            yield node.name, "function", None, node
        elif isinstance(node, ast.AsyncFunctionDef):
            yield node.name, "async_function", None, node
        elif isinstance(node, ast.ClassDef):
            for child in node.body:
                if isinstance(child, ast.FunctionDef):
                    yield f"{node.name}.{child.name}", "method", node.name, child
                elif isinstance(child, ast.AsyncFunctionDef):
                    yield f"{node.name}.{child.name}", "async_method", node.name, child


def _build_module_index(module_name: str, tree: ast.Module) -> ModuleIndex:
    index = ModuleIndex()

    for node in tree.body:
        if isinstance(node, ast.FunctionDef | ast.AsyncFunctionDef):
            index.functions[node.name] = f"{module_name}.{node.name}"

        elif isinstance(node, ast.ClassDef):
            class_full = f"{module_name}.{node.name}"
            index.classes[node.name] = class_full
            index.methods[node.name] = {}

            for child in node.body:
                if isinstance(child, ast.FunctionDef | ast.AsyncFunctionDef):
                    index.methods[node.name][child.name] = f"{module_name}.{node.name}.{child.name}"

    return index


def _build_global_indexes(
    module_trees: dict[str, ast.Module],
) -> tuple[dict[str, ModuleIndex], set[str], dict[str, set[str]]]:
    """
    Returns:
      - per-module index
      - all rich-defined full names
      - method-name lookup: method_name -> set(full_method_names)
    """
    module_indexes: dict[str, ModuleIndex] = {}
    all_full_names: set[str] = set()
    method_name_to_full_names: dict[str, set[str]] = {}

    for module_name, tree in module_trees.items():
        idx = _build_module_index(module_name, tree)
        module_indexes[module_name] = idx

        all_full_names.update(idx.functions.values())
        all_full_names.update(idx.classes.values())
        for methods in idx.methods.values():
            all_full_names.update(methods.values())
            for method_name, full_name in methods.items():
                method_name_to_full_names.setdefault(method_name, set()).add(full_name)

    return module_indexes, all_full_names, method_name_to_full_names


def _collect_import_aliases(module_name: str, tree: ast.Module) -> dict[str, str]:
    """
    Map local imported names to fully qualified module/object names.

    Examples:
      from .console import Console  -> {"Console": "rich.console.Console"}
      import rich.console as rc     -> {"rc": "rich.console"}
    """
    aliases: dict[str, str] = {}

    package_parts = module_name.split(".")
    current_pkg = package_parts[:-1]  # module's package path

    for node in tree.body:
        if isinstance(node, ast.Import):
            for alias in node.names:
                local = alias.asname or alias.name.split(".")[0]
                aliases[local] = alias.name

        elif isinstance(node, ast.ImportFrom):
            if node.module is None:
                continue

            if node.level == 0:
                base = node.module
            else:
                # resolve relative import
                prefix_parts = current_pkg[: len(current_pkg) - (node.level - 1)]
                base = ".".join(prefix_parts + [node.module])

            for alias in node.names:
                if alias.name == "*":
                    continue
                local = alias.asname or alias.name
                aliases[local] = f"{base}.{alias.name}"

    return aliases


class RichCallCollector(ast.NodeVisitor):
    def __init__(
        self,
        *,
        module_name: str,
        module_index: ModuleIndex,
        import_aliases: dict[str, str],
        all_full_names: set[str],
        method_name_to_full_names: dict[str, set[str]],
        current_class_name: str | None = None,
    ) -> None:
        self.module_name = module_name
        self.module_index = module_index
        self.import_aliases = import_aliases
        self.all_full_names = all_full_names
        self.method_name_to_full_names = method_name_to_full_names
        self.current_class_name = current_class_name
        self.calls: list[str] = []

    def visit_Call(self, node: ast.Call) -> None:
        resolved = self._resolve_call(node.func)
        if resolved is not None:
            self.calls.append(resolved)
        self.generic_visit(node)

    def _resolve_call(self, node: ast.AST) -> str | None:
        # foo()
        if isinstance(node, ast.Name):
            name = node.id

            # local function in same module
            if name in self.module_index.functions:
                return self.module_index.functions[name]

            # local class in same module
            if name in self.module_index.classes:
                return self.module_index.classes[name]

            # imported alias that points into rich.*
            target = self.import_aliases.get(name)
            if target and target.startswith("rich.") and target in self.all_full_names:
                return target

            return None

        # obj.method()
        if isinstance(node, ast.Attribute):
            # self.foo() or cls.foo() inside a Rich class
            if (
                isinstance(node.value, ast.Name)
                and node.value.id in {"self", "cls"}
                and self.current_class_name is not None
            ):
                method_map = self.module_index.methods.get(self.current_class_name, {})
                return method_map.get(node.attr)

            # rich.console.Console(...)
            dotted = self._flatten_attribute(node)
            if dotted.startswith("rich.") and dotted in self.all_full_names:
                return dotted

            # alias.attr() where alias was imported
            root_name = self._root_name(node)
            if root_name is not None and root_name in self.import_aliases:
                root_target = self.import_aliases[root_name]
                suffix = dotted[len(root_name):]
                candidate = root_target + suffix
                if candidate.startswith("rich.") and candidate in self.all_full_names:
                    return candidate

            # Heuristic: bare method like something.render() where render is unique in Rich
            candidates = self.method_name_to_full_names.get(node.attr, set())
            if len(candidates) == 1:
                return next(iter(candidates))

            return None

        return None

    def _flatten_attribute(self, node: ast.Attribute) -> str:
        parts: list[str] = []
        current: ast.AST = node

        while isinstance(current, ast.Attribute):
            parts.append(current.attr)
            current = current.value

        if isinstance(current, ast.Name):
            parts.append(current.id)
        else:
            return ""

        return ".".join(reversed(parts))

    def _root_name(self, node: ast.Attribute) -> str | None:
        current: ast.AST = node
        while isinstance(current, ast.Attribute):
            current = current.value
        if isinstance(current, ast.Name):
            return current.id
        return None


def collect_rich_only_call_graph() -> list[FunctionCallInfo]:
    module_names = list(_iter_rich_module_names())

    module_trees: dict[str, ast.Module] = {}
    skipped_no_source = 0
    failed_imports: list[str] = []

    for module_name in tqdm(module_names, desc="Parsing modules"):
        try:
            source = _module_source(module_name)
            if source is None:
                skipped_no_source += 1
                continue
            module_trees[module_name] = ast.parse(source, filename=module_name)
        except Exception:
            failed_imports.append(module_name)

    module_indexes, all_full_names, method_name_to_full_names = _build_global_indexes(module_trees)

    results: list[FunctionCallInfo] = []

    all_defs: list[tuple[str, str, str, str | None, ast.FunctionDef | ast.AsyncFunctionDef]] = []
    for module_name, tree in module_trees.items():
        for qualname, kind, class_name, node in _iter_defined_functions(tree):
            if class_name is None:
                full_name = f"{module_name}.{qualname}"
            else:
                full_name = f"{module_name}.{class_name}.{node.name}"
            all_defs.append((module_name, qualname, full_name, class_name, node))

    for module_name, qualname, full_name, class_name, node in tqdm(all_defs, desc="Resolving calls"):
        import_aliases = _collect_import_aliases(module_name, module_trees[module_name])
        collector = RichCallCollector(
            module_name=module_name,
            module_index=module_indexes[module_name],
            import_aliases=import_aliases,
            all_full_names=all_full_names,
            method_name_to_full_names=method_name_to_full_names,
            current_class_name=class_name,
        )
        collector.visit(node)

        results.append(
            FunctionCallInfo(
                module_name=module_name,
                qualname=qualname,
                full_name=full_name,
                kind="method" if class_name else "function",
                lineno=getattr(node, "lineno", None),
                end_lineno=getattr(node, "end_lineno", None),
                calls=sorted(set(collector.calls)),
            )
        )

    console = rich.get_console()
    console.rule("[bold green]Summary[/bold green]")
    console.print(f"Parsed modules: {len(module_trees)}")
    console.print(f"Collected functions/methods: {len(results)}")
    console.print(f"Modules with no source: {skipped_no_source}")
    if failed_imports:
        console.print(f"Failed imports: {failed_imports}")

    return results


if __name__ == "__main__":
    infos = collect_rich_only_call_graph()

    # console = rich.get_console()
    # for info in infos:
    #     console.rule(f"[bold cyan]{info.full_name}[/bold cyan]")
    #     console.print(info.calls)

Resolving calls: 100%|██████████| 814/814 [00:00<00:00, 14293.76it/s]


───────────────────────────────────────────────────── Summary ─────────────────────────────────────────────────────

Parsed modules: 98

Collected functions/methods: 814

Modules with no source: 0

Failed imports: ['rich._win32_console', 'rich._windows_renderer']

In [49]:
from __future__ import annotations

import ast
import importlib
import inspect
import pkgutil
from collections.abc import Iterator
from dataclasses import dataclass, field
import os
from pathlib import Path
from typing import NamedTuple

import networkx as nx
from tqdm import tqdm
import rich

RICH_GITHUB_REPO = "Textualize/rich"


class RichGithubTestRef(NamedTuple):
    """Reference to a pytest case in a Textualize/rich GitHub checkout."""

    repo_relative_path: str
    test_qualname: str
    lineno: int
    url: str


def _rich_github_blob_url(*, github_ref: str, repo_relative_path: str, lineno: int) -> str:
    return f"https://github.com/{RICH_GITHUB_REPO}/blob/{github_ref}/{repo_relative_path}#L{lineno}"


@dataclass
class FunctionCallInfo:
    module_name: str
    qualname: str                   # e.g. "get_console", "Console.print"
    full_name: str                  # e.g. "rich.get_console", "rich.console.Console.print"
    kind: str                       # function | method | async_function | async_method
    class_name: str | None
    lineno: int | None
    end_lineno: int | None
    calls: list[str] = field(default_factory=list)  # resolved rich-only callees
    github_tests: list[RichGithubTestRef] = field(
        default_factory=list,
    )  # pytest cases in a rich repo checkout that call this symbol

@dataclass
class ClassInfo:
    module_name: str
    class_name: str
    full_name: str
    lineno: int | None
    end_lineno: int | None
    methods: list[FunctionCallInfo] = field(default_factory=list)

@dataclass
class ModuleIndex:
    functions: dict[str, str] = field(default_factory=dict)      # local name -> full name
    classes: dict[str, str] = field(default_factory=dict)        # local class -> full name
    methods: dict[str, dict[str, str]] = field(default_factory=dict)  # class -> method -> full name


def _iter_rich_module_names() -> Iterator[str]:
    yield rich.__name__
    for _finder, name, _ispkg in pkgutil.walk_packages(
        path=rich.__path__,
        prefix=f"{rich.__name__}.",
    ):
        yield name

def _iter_defined_classes(
    tree: ast.Module,
) -> Iterator[ast.ClassDef]:
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            yield node

def _module_source(module_name: str) -> str | None:
    module = importlib.import_module(module_name)

    try:
        return inspect.getsource(module)
    except (OSError, TypeError):
        pass

    try:
        filename = inspect.getsourcefile(module)
        if filename is None:
            return None
        return Path(filename).read_text(encoding="utf-8")
    except OSError:
        return None


def _iter_defined_functions(
    tree: ast.Module,
) -> Iterator[tuple[str, str, str | None, ast.FunctionDef | ast.AsyncFunctionDef]]:
    """
    Yields:
      (qualname, kind, class_name, node)

    Examples:
      ("get_console", "function", None, node)
      ("Console.print", "method", "Console", node)
    """
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            yield node.name, "function", None, node
        elif isinstance(node, ast.AsyncFunctionDef):
            yield node.name, "async_function", None, node
        elif isinstance(node, ast.ClassDef):
            for child in node.body:
                if isinstance(child, ast.FunctionDef):
                    yield f"{node.name}.{child.name}", "method", node.name, child
                elif isinstance(child, ast.AsyncFunctionDef):
                    yield f"{node.name}.{child.name}", "async_method", node.name, child


def _build_module_index(module_name: str, tree: ast.Module) -> ModuleIndex:
    index = ModuleIndex()

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            index.functions[node.name] = f"{module_name}.{node.name}"

        elif isinstance(node, ast.ClassDef):
            index.classes[node.name] = f"{module_name}.{node.name}"
            index.methods[node.name] = {}

            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    index.methods[node.name][child.name] = f"{module_name}.{node.name}.{child.name}"

    return index


def _build_global_indexes(
    module_trees: dict[str, ast.Module],
) -> tuple[dict[str, ModuleIndex], set[str], dict[str, set[str]]]:
    module_indexes: dict[str, ModuleIndex] = {}
    all_full_names: set[str] = set()
    method_name_to_full_names: dict[str, set[str]] = {}

    for module_name, tree in module_trees.items():
        idx = _build_module_index(module_name, tree)
        module_indexes[module_name] = idx

        all_full_names.update(idx.functions.values())
        all_full_names.update(idx.classes.values())

        for methods in idx.methods.values():
            all_full_names.update(methods.values())
            for method_name, full_name in methods.items():
                method_name_to_full_names.setdefault(method_name, set()).add(full_name)

    return module_indexes, all_full_names, method_name_to_full_names


def _collect_import_aliases(module_name: str, tree: ast.Module) -> dict[str, str]:
    """
    Build local import alias -> fully-qualified target.

    Examples:
      from .console import Console  -> {"Console": "rich.console.Console"}
      import rich.console as rc     -> {"rc": "rich.console"}
    """
    aliases: dict[str, str] = {}

    package_parts = module_name.split(".")
    current_pkg = package_parts[:-1]

    for node in tree.body:
        if isinstance(node, ast.Import):
            for alias in node.names:
                local = alias.asname or alias.name.split(".")[0]
                aliases[local] = alias.name

        elif isinstance(node, ast.ImportFrom):
            if node.module is None:
                continue

            if node.level == 0:
                base = node.module
            else:
                prefix_parts = current_pkg[: len(current_pkg) - (node.level - 1)]
                base = ".".join(prefix_parts + [node.module])

            for alias in node.names:
                if alias.name == "*":
                    continue
                local = alias.asname or alias.name
                aliases[local] = f"{base}.{alias.name}"

    return aliases


def _parse_tests_module_dotted_name(repo_relative_path: str) -> str:
    path = Path(repo_relative_path)
    parts = list(path.with_suffix("").parts)
    if parts and parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts) if parts else "tests"


def _iter_pytest_methods_in_class(
    body: list[ast.stmt],
    *,
    class_short_name: str,
    qualname_prefix: str,
) -> Iterator[tuple[str, str | None, ast.FunctionDef | ast.AsyncFunctionDef]]:
    for node in body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name.startswith("test"):
                yield f"{qualname_prefix}.{node.name}", class_short_name, node
        elif isinstance(node, ast.ClassDef) and node.name.startswith("Test"):
            yield from _iter_pytest_methods_in_class(
                node.body,
                class_short_name=node.name,
                qualname_prefix=f"{qualname_prefix}.{node.name}",
            )


def _iter_pytest_test_nodes(
    body: list[ast.stmt],
    *,
    outer_class_qualname: str,
) -> Iterator[tuple[str, str | None, ast.FunctionDef | ast.AsyncFunctionDef]]:
    for node in body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name.startswith("test"):
                qual = f"{outer_class_qualname}.{node.name}" if outer_class_qualname else node.name
                yield qual, None, node
        elif isinstance(node, ast.ClassDef) and node.name.startswith("Test"):
            prefix = f"{outer_class_qualname}.{node.name}" if outer_class_qualname else node.name
            yield from _iter_pytest_methods_in_class(
                node.body,
                class_short_name=node.name,
                qualname_prefix=prefix,
            )


def _iter_pytest_module_tests(
    tree: ast.Module,
) -> Iterator[tuple[str, str | None, ast.FunctionDef | ast.AsyncFunctionDef]]:
    yield from _iter_pytest_test_nodes(tree.body, outer_class_qualname="")


_GITHUB_TEST_INDEX_CACHE: dict[tuple[str, str], dict[str, list[RichGithubTestRef]]] = {}


def _get_cached_github_test_index(
    *,
    rich_repo_root: Path | None,
    all_full_names: set[str],
    method_name_to_full_names: dict[str, set[str]],
    github_ref: str,
) -> dict[str, list[RichGithubTestRef]]:
    if rich_repo_root is None or not rich_repo_root.is_dir():
        return {}
    cache_key = (str(rich_repo_root.resolve()), github_ref)
    if cache_key not in _GITHUB_TEST_INDEX_CACHE:
        _GITHUB_TEST_INDEX_CACHE[cache_key] = _build_github_test_index(
            rich_repo_root=rich_repo_root,
            all_full_names=all_full_names,
            method_name_to_full_names=method_name_to_full_names,
            github_ref=github_ref,
        )
    return _GITHUB_TEST_INDEX_CACHE[cache_key]


def _build_github_test_index(
    *,
    rich_repo_root: Path,
    all_full_names: set[str],
    method_name_to_full_names: dict[str, set[str]],
    github_ref: str,
) -> dict[str, list[RichGithubTestRef]]:
    """
    Map each rich production full_name to pytest tests that (statically) call it.

    Requires a local clone of https://github.com/Textualize/rich with a tests/ tree.
    """
    tests_root = rich_repo_root / "tests"
    if not tests_root.is_dir():
        return {}

    reverse: dict[str, set[RichGithubTestRef]] = {}
    paths = sorted(tests_root.rglob("*.py"))

    for path in tqdm(paths, desc="Indexing rich GitHub tests"):
        rel = path.relative_to(rich_repo_root).as_posix()
        try:
            source = path.read_text(encoding="utf-8")
        except OSError:
            continue
        try:
            tree = ast.parse(source, filename=rel)
        except SyntaxError:
            continue

        mod_dotted = _parse_tests_module_dotted_name(rel)
        module_index = _build_module_index(mod_dotted, tree)
        import_aliases = _collect_import_aliases(mod_dotted, tree)

        for qual, cls_name, test_node in _iter_pytest_module_tests(tree):
            line = test_node.lineno or 1
            ref = RichGithubTestRef(
                rel,
                qual,
                line,
                _rich_github_blob_url(
                    github_ref=github_ref,
                    repo_relative_path=rel,
                    lineno=line,
                ),
            )
            collector = RichCallCollector(
                module_name=mod_dotted,
                module_index=module_index,
                import_aliases=import_aliases,
                all_full_names=all_full_names,
                method_name_to_full_names=method_name_to_full_names,
                current_class_name=cls_name,
            )
            collector.visit(test_node)
            for sym in set(collector.calls):
                reverse.setdefault(sym, set()).add(ref)

    return {
        sym: sorted(refs, key=lambda r: (r.repo_relative_path, r.lineno, r.test_qualname))
        for sym, refs in reverse.items()
    }


class RichCallCollector(ast.NodeVisitor):
    def __init__(
        self,
        *,
        module_name: str,
        module_index: ModuleIndex,
        import_aliases: dict[str, str],
        all_full_names: set[str],
        method_name_to_full_names: dict[str, set[str]],
        current_class_name: str | None = None,
    ) -> None:
        self.module_name = module_name
        self.module_index = module_index
        self.import_aliases = import_aliases
        self.all_full_names = all_full_names
        self.method_name_to_full_names = method_name_to_full_names
        self.current_class_name = current_class_name
        self.calls: list[str] = []

    def visit_Call(self, node: ast.Call) -> None:
        resolved = self._resolve_call(node.func)
        if resolved is not None:
            self.calls.append(resolved)
        self.generic_visit(node)

    def _resolve_call(self, node: ast.AST) -> str | None:
        # foo()
        if isinstance(node, ast.Name):
            name = node.id

            if name in self.module_index.functions:
                return self.module_index.functions[name]

            if name in self.module_index.classes:
                return self.module_index.classes[name]

            target = self.import_aliases.get(name)
            if target and target.startswith("rich.") and target in self.all_full_names:
                return target

            return None

        # obj.method()
        if isinstance(node, ast.Attribute):
            # self.foo() or cls.foo()
            if (
                isinstance(node.value, ast.Name)
                and node.value.id in {"self", "cls"}
                and self.current_class_name is not None
            ):
                method_map = self.module_index.methods.get(self.current_class_name, {})
                return method_map.get(node.attr)

            dotted = self._flatten_attribute(node)
            if dotted.startswith("rich.") and dotted in self.all_full_names:
                return dotted

            root_name = self._root_name(node)
            if root_name is not None and root_name in self.import_aliases:
                root_target = self.import_aliases[root_name]
                suffix = dotted[len(root_name):]
                candidate = root_target + suffix
                if candidate.startswith("rich.") and candidate in self.all_full_names:
                    return candidate

            # heuristic: only resolve bare attribute methods if unique in entire package
            candidates = self.method_name_to_full_names.get(node.attr, set())
            if len(candidates) == 1:
                return next(iter(candidates))

            return None

        return None

    def _flatten_attribute(self, node: ast.Attribute) -> str:
        parts: list[str] = []
        current: ast.AST = node

        while isinstance(current, ast.Attribute):
            parts.append(current.attr)
            current = current.value

        if isinstance(current, ast.Name):
            parts.append(current.id)
        else:
            return ""

        return ".".join(reversed(parts))

    def _root_name(self, node: ast.Attribute) -> str | None:
        current: ast.AST = node
        while isinstance(current, ast.Attribute):
            current = current.value
        if isinstance(current, ast.Name):
            return current.id
        return None


def collect_rich_class_infos(
    *,
    rich_repo_root: Path | None = None,
    github_ref: str = "main",
) -> list[ClassInfo]:
    module_names = list(_iter_rich_module_names())

    module_trees: dict[str, ast.Module] = {}
    skipped_no_source = 0
    failed_imports: list[str] = []

    for module_name in tqdm(module_names, desc="Parsing modules"):
        try:
            source = _module_source(module_name)
            if source is None:
                skipped_no_source += 1
                continue
            module_trees[module_name] = ast.parse(source, filename=module_name)
        except Exception:
            failed_imports.append(module_name)

    module_indexes, all_full_names, method_name_to_full_names = _build_global_indexes(module_trees)

    test_by_symbol = _get_cached_github_test_index(
        rich_repo_root=rich_repo_root,
        all_full_names=all_full_names,
        method_name_to_full_names=method_name_to_full_names,
        github_ref=github_ref,
    )

    results: list[ClassInfo] = []

    all_classes: list[tuple[str, ast.ClassDef]] = []
    for module_name, tree in module_trees.items():
        for class_node in _iter_defined_classes(tree):
            all_classes.append((module_name, class_node))

    for module_name, class_node in tqdm(all_classes, desc="Resolving class methods"):
        class_name = class_node.name
        class_full_name = f"{module_name}.{class_name}"
        import_aliases = _collect_import_aliases(module_name, module_trees[module_name])

        methods: list[FunctionCallInfo] = []

        for child in class_node.body:
            if not isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                continue

            qualname = f"{class_name}.{child.name}"
            full_name = f"{module_name}.{class_name}.{child.name}"
            kind = "async_method" if isinstance(child, ast.AsyncFunctionDef) else "method"

            collector = RichCallCollector(
                module_name=module_name,
                module_index=module_indexes[module_name],
                import_aliases=import_aliases,
                all_full_names=all_full_names,
                method_name_to_full_names=method_name_to_full_names,
                current_class_name=class_name,
            )
            collector.visit(child)

            methods.append(
                FunctionCallInfo(
                    module_name=module_name,
                    qualname=qualname,
                    full_name=full_name,
                    kind=kind,
                    class_name=class_name,
                    lineno=getattr(child, "lineno", None),
                    end_lineno=getattr(child, "end_lineno", None),
                    calls=sorted(set(collector.calls)),
                    github_tests=list(test_by_symbol.get(full_name, [])),
                )
            )

        results.append(
            ClassInfo(
                module_name=module_name,
                class_name=class_name,
                full_name=class_full_name,
                lineno=getattr(class_node, "lineno", None),
                end_lineno=getattr(class_node, "end_lineno", None),
                methods=methods,
            )
        )

    console = rich.get_console()
    console.rule("[bold green]Summary[/bold green]")
    console.print(f"Parsed modules: {len(module_trees)}")
    console.print(f"Collected functions/methods: {len(results)}")
    console.print(f"Modules with no source: {skipped_no_source}")
    if failed_imports:
        console.print(f"Failed imports: {failed_imports}")
    if rich_repo_root is not None and rich_repo_root.is_dir():
        n_links = sum(len(v) for v in test_by_symbol.values())
        console.print(
            f"GitHub test index: {len(test_by_symbol)} rich symbols with tests, {n_links} total links",
        )

    return results

def _sanitize_str(value):
    return "" if value is None else str(value)


def _sanitize_int(value):
    return -1 if value is None else int(value)

def collect_rich_only_call_infos(
    *,
    rich_repo_root: Path | None = None,
    github_ref: str = "main",
) -> list[FunctionCallInfo]:
    module_names = list(_iter_rich_module_names())

    module_trees: dict[str, ast.Module] = {}
    skipped_no_source = 0
    failed_imports: list[str] = []

    for module_name in tqdm(module_names, desc="Parsing modules"):
        try:
            source = _module_source(module_name)
            if source is None:
                skipped_no_source += 1
                continue
            module_trees[module_name] = ast.parse(source, filename=module_name)
        except Exception:
            failed_imports.append(module_name)

    module_indexes, all_full_names, method_name_to_full_names = _build_global_indexes(module_trees)

    test_by_symbol = _get_cached_github_test_index(
        rich_repo_root=rich_repo_root,
        all_full_names=all_full_names,
        method_name_to_full_names=method_name_to_full_names,
        github_ref=github_ref,
    )

    all_defs: list[tuple[str, str, str, str | None, str, ast.FunctionDef | ast.AsyncFunctionDef]] = []
    for module_name, tree in module_trees.items():
        for qualname, kind, class_name, node in _iter_defined_functions(tree):
            if class_name is None:
                full_name = f"{module_name}.{qualname}"
            else:
                full_name = f"{module_name}.{class_name}.{node.name}"
            all_defs.append((module_name, qualname, full_name, class_name, kind, node))

    results: list[FunctionCallInfo] = []

    for module_name, qualname, full_name, class_name, kind, node in tqdm(all_defs, desc="Resolving calls"):
        import_aliases = _collect_import_aliases(module_name, module_trees[module_name])

        collector = RichCallCollector(
            module_name=module_name,
            module_index=module_indexes[module_name],
            import_aliases=import_aliases,
            all_full_names=all_full_names,
            method_name_to_full_names=method_name_to_full_names,
            current_class_name=class_name,
        )
        collector.visit(node)

        results.append(
            FunctionCallInfo(
                module_name=module_name,
                qualname=qualname,
                full_name=full_name,
                kind=kind,
                class_name=class_name,
                lineno=getattr(node, "lineno", None),
                end_lineno=getattr(node, "end_lineno", None),
                calls=sorted(set(collector.calls)),
                github_tests=list(test_by_symbol.get(full_name, [])),
            )
        )

    console = rich.get_console()
    console.rule("[bold green]Summary[/bold green]")
    console.print(f"Parsed modules: {len(module_trees)}")
    console.print(f"Collected functions/methods: {len(results)}")
    console.print(f"Modules with no source: {skipped_no_source}")
    if failed_imports:
        console.print(f"Failed imports: {failed_imports}")
    if rich_repo_root is not None and rich_repo_root.is_dir():
        n_links = sum(len(v) for v in test_by_symbol.values())
        console.print(
            f"GitHub test index: {len(test_by_symbol)} rich symbols with tests, {n_links} total links",
        )

    return results


def github_tests(function_info: FunctionCallInfo) -> list[RichGithubTestRef]:
    """Return GitHub test references that call a specific rich symbol."""
    return function_info.github_tests


def build_call_graph(
    infos: list[FunctionCallInfo],
    class_infos: list[ClassInfo],
) -> nx.DiGraph:
    """
    Build a directed graph with:
      - class nodes
      - function/method nodes
      - edges:
          class -> method        (structure)
          caller -> callee       (calls)
    """
    G = nx.DiGraph()

    # -----------------------------
    # 1. Add class nodes
    # -----------------------------
    for cls in class_infos:
        G.add_node(
            cls.full_name,
            node_type="class",
            module=_sanitize_str(cls.module_name),
            class_name=_sanitize_str(cls.class_name),
            lineno=_sanitize_int(cls.lineno),
            end_lineno=_sanitize_int(cls.end_lineno),
        )

    # -----------------------------
    # 2. Add function/method nodes
    # -----------------------------
    for info in infos:
        G.add_node(
            info.full_name,
            node_type=info.kind,  # "function", "method", etc.
            module=_sanitize_str(info.module_name),
            qualname=_sanitize_str(info.qualname),
            class_name=_sanitize_str(info.class_name),
            lineno=_sanitize_int(info.lineno),
            end_lineno=_sanitize_int(info.end_lineno),
        )

        # -----------------------------
        # 3. Add class → method edges
        # -----------------------------
        if info.class_name is not None:
            class_full = f"{info.module_name}.{info.class_name}"

            if not G.has_node(class_full):
                # fallback safety (should already exist)
                G.add_node(
                    class_full,
                    node_type="class",
                    module=_sanitize_str(info.module_name),
                    class_name=_sanitize_str(info.class_name),
                    lineno=-1,
                    end_lineno=-1,
                )

            G.add_edge(
                class_full,
                info.full_name,
                edge_type="has_method",
            )

    # -----------------------------
    # 4. Add call edges
    # -----------------------------
    for info in infos:
        for callee in info.calls:
            if not G.has_node(callee):
                # optional: skip or add external node
                G.add_node(callee, node_type="external")

            G.add_edge(
                info.full_name,
                callee,
                edge_type="calls",
            )

    return G

def print_graph_summary(G: nx.DiGraph) -> None:
    console = rich.get_console()
    console.rule("[bold blue]Graph summary[/bold blue]")
    console.print(f"Nodes: {G.number_of_nodes()}")
    console.print(f"Edges: {G.number_of_edges()}")

    in_degrees = sorted(G.in_degree, key=lambda x: x[1], reverse=True)[:10]
    out_degrees = sorted(G.out_degree, key=lambda x: x[1], reverse=True)[:10]

    console.print("\n[bold]Top 10 most-called nodes (highest in-degree)[/bold]")
    for node, deg in in_degrees:
        console.print(f"{deg:>3}  {node}")

    console.print("\n[bold]Top 10 most-calling nodes (highest out-degree)[/bold]")
    for node, deg in out_degrees:
        console.print(f"{deg:>3}  {node}")


def export_graph(G: nx.DiGraph, path: str):
    try:
        nx.write_graphml(G, path)
    except TypeError as e:
        print("GraphML export failed, retrying without attributes:", e)

        G_clean = nx.DiGraph()
        G_clean.add_edges_from(G.edges())
        nx.write_graphml(G_clean, path)

if __name__ == "__main__":
    _env_root = os.environ.get("RICH_REPO_ROOT", "").strip()
    _repo = Path(_env_root).expanduser().resolve() if _env_root else None
    function_infos = collect_rich_only_call_infos(rich_repo_root=_repo)
    class_infos = collect_rich_class_infos(rich_repo_root=_repo)

    G = build_call_graph(function_infos, class_infos)

Resolving calls: 100%|██████████| 814/814 [00:00<00:00, 13975.69it/s]


───────────────────────────────────────────────────── Summary ─────────────────────────────────────────────────────

Parsed modules: 98

Collected functions/methods: 814

Modules with no source: 0

...

───────────────────────────────────────────────────── Summary ─────────────────────────────────────────────────────

Parsed modules: 98

Collected functions/methods: 168

Modules with no source: 0

Failed imports: ['rich._win32_console', 'rich._windows_renderer']

In [51]:
for inf in function_infos:
    tests = github_tests(inf)
    if tests:
        print(inf.full_name)
        print(tests[0])
        break


In [39]:
class_infos[0]

ClassInfo(module_name='rich.__main__', class_name='ColorBox', full_name='rich.__main__.ColorBox', lineno=18, end_lineno=36, methods=[FunctionCallInfo(module_name='rich.__main__', qualname='ColorBox.__rich_console__', full_name='rich.__main__.ColorBox.__rich_console__', kind='method', class_name='ColorBox', lineno=19, end_lineno=31, calls=['rich.color.Color.from_rgb', 'rich.segment.Segment', 'rich.segment.Segment.line', 'rich.style.Style']), FunctionCallInfo(module_name='rich.__main__', qualname='ColorBox.__rich_measure__', full_name='rich.__main__.ColorBox.__rich_measure__', kind='method', class_name='ColorBox', lineno=33, end_lineno=36, calls=['rich.measure.Measurement'])])

In [40]:
for inf in class_infos:
    if inf.methods:
        print(inf)
        break

ClassInfo(module_name='rich.__main__', class_name='ColorBox', full_name='rich.__main__.ColorBox', lineno=18, end_lineno=36, methods=[FunctionCallInfo(module_name='rich.__main__', qualname='ColorBox.__rich_console__', full_name='rich.__main__.ColorBox.__rich_console__', kind='method', class_name='ColorBox', lineno=19, end_lineno=31, calls=['rich.color.Color.from_rgb', 'rich.segment.Segment', 'rich.segment.Segment.line', 'rich.style.Style']), FunctionCallInfo(module_name='rich.__main__', qualname='ColorBox.__rich_measure__', full_name='rich.__main__.ColorBox.__rich_measure__', kind='method', class_name='ColorBox', lineno=33, end_lineno=36, calls=['rich.measure.Measurement'])])


In [41]:
def build_call_only_graph(infos: list[FunctionCallInfo]) -> nx.DiGraph:
    """
    Graph with only call relationships:
      caller -> callee
    """
    G = nx.DiGraph()

    for info in infos:
        G.add_node(
            info.full_name,
            node_type=info.kind,
            module=info.module_name,
            qualname=info.qualname,
            class_name="" if info.class_name is None else info.class_name,
            lineno=-1 if info.lineno is None else info.lineno,
            end_lineno=-1 if info.end_lineno is None else info.end_lineno,
        )

    for info in infos:
        for callee in info.calls:
            if not G.has_node(callee):
                G.add_node(callee, node_type="external")
            G.add_edge(info.full_name, callee)

    return G

def compute_pagerank(
    G: nx.DiGraph,
    alpha: float = 0.85,
) -> dict[str, float]:
    return nx.pagerank(G, alpha=alpha)

def top_ranked_methods(
    G: nx.DiGraph,
    pagerank_scores: dict[str, float],
    top_n: int = 20,
) -> list[tuple[str, float]]:
    method_nodes = [
        node
        for node, attrs in G.nodes(data=True)
        if attrs.get("node_type") in {"method", "async_method"}
    ]

    ranked = sorted(
        ((node, pagerank_scores[node]) for node in method_nodes),
        key=lambda x: x[1],
        reverse=True,
    )
    return ranked[:top_n]

def top_ranked_classes(
    G: nx.DiGraph,
    pagerank_scores: dict[str, float],
    top_n: int = 20,
) -> list[tuple[str, float]]:
    class_nodes = [
        node
        for node, attrs in G.nodes(data=True)
        if attrs.get("node_type") == "class"
    ]

    ranked = sorted(
        ((node, pagerank_scores[node]) for node in class_nodes),
        key=lambda x: x[1],
        reverse=True,
    )
    return ranked[:top_n]

def aggregate_method_pagerank_by_class(
    infos: list[FunctionCallInfo],
    pagerank_scores: dict[str, float],
) -> dict[str, float]:
    class_scores: dict[str, float] = {}

    for info in infos:
        if info.class_name is None:
            continue

        class_full = f"{info.module_name}.{info.class_name}"
        class_scores.setdefault(class_full, 0.0)
        class_scores[class_full] += pagerank_scores.get(info.full_name, 0.0)

    return class_scores

def top_classes_by_method_pagerank(
    infos: list[FunctionCallInfo],
    pagerank_scores: dict[str, float],
    top_n: int = 20,
) -> list[tuple[str, float]]:
    class_scores = aggregate_method_pagerank_by_class(infos, pagerank_scores)
    return sorted(class_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

In [43]:
if __name__ == "__main__":
    function_infos = collect_rich_only_call_infos()
    class_infos = collect_rich_class_infos()

    # Best graph for PageRank on call centrality
    G_calls = build_call_only_graph(function_infos)

    pr = compute_pagerank(G_calls)

    print("\nTop methods by PageRank")
    for name, score in top_ranked_methods(G_calls, pr, top_n=20):
        print(f"{score:.6f}  {name}")

    print("\nTop classes by summed method PageRank")
    for name, score in top_classes_by_method_pagerank(function_infos, pr, top_n=20):
        print(f"{score:.6f}  {name}")

Parsing modules:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving calls: 100%|██████████| 814/814 [00:00<00:00, 14183.90it/s]


───────────────────────────────────────────────────── Summary ─────────────────────────────────────────────────────

Parsed modules: 98

Collected functions/methods: 814

...

Parsed modules: 98

Collected functions/methods: 168

Modules with no source: 0

Failed imports: ['rich._win32_console', 'rich._windows_renderer']


Top methods by PageRank
0.014579  rich.text.Text.rstrip
0.013453  rich.text.Text.join
0.009980  rich.console.Console.get_style
0.009078  rich.console.Console.print
0.007858  rich.pretty.Node.iter_tokens
0.006598  rich.style.Style.parse
0.006485  rich.text.Text.blank_copy
0.005744  rich.console.ConsoleOptions.copy
0.005555  rich.emoji.Emoji.replace
0.005050  rich.style.Style.null
0.004998  rich.text.Text.stylize
0.004780  rich.markdown.Markdown._flatten_tokens
0.004224  rich.measure.Measurement.get
0.003938  rich.text.Text.from_markup
0.003470  rich.progress.Progress.advance
0.003200  rich.padding.Padding.unpack
0.003119  rich.color.Color.parse
0.003084  rich.measure.Measurement.span
0.002885  rich.console.Console._enter_buffer
0.002854  rich.segment.Segment.line

Top classes by summed method PageRank
0.091142  rich.text.Text
0.090023  rich.console.Console
0.039968  rich.style.Style
0.034364  rich.progress.Progress
0.027208  rich.segment.Segment
0.018190  rich.table.Table
0.015922  ric

In [44]:
def top_methods_by_in_degree(
    G: nx.DiGraph,
    top_n: int = 20,
) -> list[tuple[str, int]]:
    method_nodes = [
        node
        for node, attrs in G.nodes(data=True)
        if attrs.get("node_type") in {"method", "async_method"}
    ]

    ranked = sorted(
        ((node, G.in_degree(node)) for node in method_nodes),
        key=lambda x: x[1],
        reverse=True,
    )
    return ranked[:top_n]

In [45]:
print("\nTop methods by raw call count")
for name, count in top_methods_by_in_degree(G_calls, top_n=20):
    print(f"{count:>4}  {name}")


Top methods by raw call count
  36  rich.text.Text.join
  22  rich.console.Console.print
  20  rich.console.Console.get_style
  13  rich.console.Console.render_lines
  13  rich.text.Text.from_markup
  13  rich.text.Text.stylize
  12  rich.segment.Segment.line
  11  rich.text.Text.assemble
  10  rich.style.Style.null
   9  rich.measure.Measurement.get
   8  rich.console.ConsoleOptions.update_width
   8  rich.emoji.Emoji.replace
   8  rich.padding.Padding.unpack
   8  rich.table.Table.grid
   8  rich.table.Table.add_row
   8  rich.text.Text.rstrip
   7  rich.table.Table.add_column
   6  rich.console.Console.render_str
   6  rich.console.Console.control
   6  rich.measure.Measurement.span


In [46]:
def discover_top_functions(
    G: nx.DiGraph,
    pagerank_scores: dict[str, float],
    top_n: int = 20,
) -> list[tuple[str, float]]:
    candidates = [
        (node, score)
        for node, score in pagerank_scores.items()
        if G.nodes[node].get("node_type") in {"function", "method", "async_function", "async_method"}
    ]
    return sorted(candidates, key=lambda x: x[1], reverse=True)[:top_n]

In [47]:
top_funcs = discover_top_functions(G_calls, pr, top_n=20)
for name, score in top_funcs:
    print(f"{score:.6f}  {name}")

0.014579  rich.text.Text.rstrip
0.013453  rich.text.Text.join
0.012825  rich.console.get_windows_console_features
0.009980  rich.console.Console.get_style
0.009078  rich.console.Console.print
0.008856  rich._unicode_data.load
0.008243  rich._loop.loop_last
0.007858  rich.pretty.Node.iter_tokens
0.006641  rich.cells.cell_len
0.006598  rich.style.Style.parse
0.006485  rich.text.Text.blank_copy
0.006475  rich.cells._cell_len
0.005913  rich._emoji_replace._emoji_replace
0.005744  rich.console.ConsoleOptions.copy
0.005555  rich.emoji.Emoji.replace
0.005050  rich.style.Style.null
0.004998  rich.text.Text.stylize
0.004786  rich.cells.get_character_cell_size
0.004780  rich.markdown.Markdown._flatten_tokens
0.004441  rich._unicode_data._parse_version
